# Regional project progress

Build one regional view that keeps the current project-status map available and, when Gold contains an earlier DPWH snapshot, compares the current snapshot with the immediately preceding snapshot.

This notebook reads Gold and the governed regional investment view only. It does not change Bronze, Silver, or Gold. `ACTIVE` includes projects such as `For Procurement`, so it means an approved status group and not necessarily construction in progress. The historical contract is proposed as D-55 and requires team approval before merge.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `05-analytics`;

-- current analytics must describe exactly one selected dpwh snapshot
SELECT ASSERT_TRUE(
    COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT_IF(source_snapshot_id IS NULL) = 0,
    'STOP: current Gold project rows must contain exactly one non-null DPWH snapshot.'
)
FROM `03-gold`.fact_project_snapshot
WHERE is_current_snapshot = TRUE;

## Build the view

**What:** Select the current and immediately preceding DPWH snapshots, aggregate both to official regions, and compare the same `project_key` across them.

**Why:** the existing delivery view answers the current status mix. This view adds genuine snapshot-over-snapshot change without treating `infra_year` as history or multiplying the project grain. With no prior snapshot, current status remains usable and every comparison measure stays `NULL`.

In [ ]:
%sql
CREATE OR REPLACE VIEW `05-analytics`.analytics_region_project_progress (
    region_key COMMENT 'Current official region key. Joins to dim_region.region_key. Key 0 is never in this view.',
    psgc_region_code COMMENT 'Stable official PSGC region code used to compare region rows across versions.',
    region_name COMMENT 'Official current PSGC region name.',
    region_short_name COMMENT 'Short current region label for charts.',
    source_coverage_status COMMENT 'Current coverage status from analytics_regional_investment.',
    comparison_status COMMENT 'COMPARABLE when a previous Gold snapshot exists, otherwise NO_PRIOR_SNAPSHOT.',
    current_source_snapshot_id COMMENT 'Current DPWH source snapshot.',
    current_snapshot_date_key COMMENT 'Current DPWH snapshot date as yyyymmdd.',
    previous_source_snapshot_id COMMENT 'Immediately preceding DPWH source snapshot. NULL when Gold has no earlier snapshot.',
    previous_snapshot_date_key COMMENT 'Immediately preceding DPWH snapshot date as yyyymmdd. NULL when unavailable.',
    current_project_count COMMENT 'Current projects mapped to the region.',
    current_finished_count COMMENT 'Current projects in the approved FINISHED status group.',
    current_active_count COMMENT 'Current projects in the approved ACTIVE status group. ACTIVE includes For Procurement and does not necessarily mean construction is under way.',
    current_inactive_count COMMENT 'Current projects in the approved INACTIVE status group.',
    current_unmapped_status_count COMMENT 'Current projects without an approved status group.',
    current_finished_pct COMMENT 'Current FINISHED projects divided by current region projects, times 100.',
    current_active_pct COMMENT 'Current ACTIVE projects divided by current region projects, times 100.',
    current_inactive_pct COMMENT 'Current INACTIVE projects divided by current region projects, times 100.',
    current_unmapped_status_pct COMMENT 'Current projects without an approved status group divided by current region projects, times 100.',
    current_total_reported_budget_pesos COMMENT 'Current reported project or contract budget in pesos. Not payment.',
    previous_project_count COMMENT 'Previous projects mapped to the region. NULL when no previous snapshot exists.',
    previous_finished_count COMMENT 'Previous projects in FINISHED. NULL when no previous snapshot exists.',
    previous_active_count COMMENT 'Previous projects in ACTIVE. NULL when no previous snapshot exists.',
    previous_inactive_count COMMENT 'Previous projects in INACTIVE. NULL when no previous snapshot exists.',
    previous_unmapped_status_count COMMENT 'Previous projects without an approved status group. NULL when no previous snapshot exists.',
    previous_total_reported_budget_pesos COMMENT 'Previous reported budget in pesos. NULL when no previous snapshot exists or no previous project has a reported budget.',
    project_count_change COMMENT 'Current project count minus previous project count. NULL without a previous snapshot.',
    reported_budget_change_pesos COMMENT 'Current reported budget minus previous reported budget. NULL without a previous snapshot or when either side has no reported budget.',
    status_changed_count COMMENT 'Projects present in both snapshots whose approved status group changed. Attributed to the current region.',
    moved_to_finished_count COMMENT 'Projects present in both snapshots that moved from a non-FINISHED group to FINISHED. Attributed to the current region.',
    progress_increased_count COMMENT 'Projects present in both snapshots whose valid physical progress increased. Attributed to the current region.',
    progress_decreased_count COMMENT 'Projects present in both snapshots whose valid physical progress decreased. Review as a source change, not automatically as an error.',
    new_project_count COMMENT 'Projects present only in the current snapshot, attributed to the current region.',
    no_longer_present_count COMMENT 'Projects present only in the previous snapshot, attributed to the previous region.',
    region_changed_count COMMENT 'Projects present in both snapshots whose official PSGC region code changed, attributed to the current region.'
)
COMMENT 'Regional current status and immediately previous snapshot comparison (D-55, proposed). One row per current official PSGC region. Current status works with one snapshot. Historical measures are NULL until a prior snapshot exists.'
AS
WITH snapshot_catalog AS (
    SELECT
        source_snapshot_id,
        MAX(snapshot_date_key) AS snapshot_date_key,
        MAX(source_load_ts) AS source_load_ts,
        MAX(CASE WHEN is_current_snapshot THEN 1 ELSE 0 END) AS is_current_snapshot
    FROM `03-gold`.fact_project_snapshot
    GROUP BY source_snapshot_id
),

current_snapshot AS (
    SELECT source_snapshot_id, snapshot_date_key
    FROM snapshot_catalog
    WHERE is_current_snapshot = 1
),

previous_snapshot AS (
    SELECT source_snapshot_id, snapshot_date_key
    FROM snapshot_catalog
    WHERE is_current_snapshot = 0
    QUALIFY ROW_NUMBER() OVER (
        ORDER BY snapshot_date_key DESC, source_load_ts DESC, source_snapshot_id DESC
    ) = 1
),

snapshot_context AS (
    SELECT
        current.source_snapshot_id AS current_source_snapshot_id,
        current.snapshot_date_key AS current_snapshot_date_key,
        previous.source_snapshot_id AS previous_source_snapshot_id,
        previous.snapshot_date_key AS previous_snapshot_date_key
    FROM current_snapshot AS current
    LEFT JOIN previous_snapshot AS previous
        ON TRUE
),

current_projects AS (
    SELECT
        fact.project_key,
        fact.region_key,
        region.psgc_region_code,
        status.status_group,
        fact.physical_progress_pct,
        fact.reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    INNER JOIN `03-gold`.dim_region AS region
        ON fact.region_key = region.region_key
    INNER JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    WHERE fact.is_current_snapshot = TRUE
        AND fact.region_key <> 0
),

previous_projects AS (
    SELECT
        fact.project_key,
        fact.region_key,
        region.psgc_region_code,
        status.status_group,
        fact.physical_progress_pct,
        fact.reported_budget_pesos
    FROM `03-gold`.fact_project_snapshot AS fact
    INNER JOIN snapshot_context AS context
        ON fact.source_snapshot_id = context.previous_source_snapshot_id
    INNER JOIN `03-gold`.dim_region AS region
        ON fact.region_key = region.region_key
    INNER JOIN `03-gold`.dim_project_status AS status
        ON fact.status_key = status.status_key
    WHERE fact.region_key <> 0
),

current_summary AS (
    SELECT
        psgc_region_code,
        COUNT(*) AS project_count,
        COUNT_IF(status_group = 'FINISHED') AS finished_count,
        COUNT_IF(status_group = 'ACTIVE') AS active_count,
        COUNT_IF(status_group = 'INACTIVE') AS inactive_count,
        COUNT_IF(status_group IS NULL) AS unmapped_status_count,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM current_projects
    GROUP BY psgc_region_code
),

previous_summary AS (
    SELECT
        psgc_region_code,
        COUNT(*) AS project_count,
        COUNT_IF(status_group = 'FINISHED') AS finished_count,
        COUNT_IF(status_group = 'ACTIVE') AS active_count,
        COUNT_IF(status_group = 'INACTIVE') AS inactive_count,
        COUNT_IF(status_group IS NULL) AS unmapped_status_count,
        SUM(reported_budget_pesos) AS total_reported_budget_pesos
    FROM previous_projects
    GROUP BY psgc_region_code
),

project_changes AS (
    SELECT
        COALESCE(current.psgc_region_code, previous.psgc_region_code) AS reporting_region_code,
        COUNT_IF(
            current.project_key IS NOT NULL
            AND previous.project_key IS NOT NULL
            AND NOT (current.status_group <=> previous.status_group)
        ) AS status_changed_count,
        COUNT_IF(
            current.project_key IS NOT NULL
            AND previous.project_key IS NOT NULL
            AND current.status_group = 'FINISHED'
            AND NOT (previous.status_group <=> 'FINISHED')
        ) AS moved_to_finished_count,
        COUNT_IF(
            current.project_key IS NOT NULL
            AND previous.project_key IS NOT NULL
            AND current.physical_progress_pct > previous.physical_progress_pct
        ) AS progress_increased_count,
        COUNT_IF(
            current.project_key IS NOT NULL
            AND previous.project_key IS NOT NULL
            AND current.physical_progress_pct < previous.physical_progress_pct
        ) AS progress_decreased_count,
        COUNT_IF(current.project_key IS NOT NULL AND previous.project_key IS NULL) AS new_project_count,
        COUNT_IF(current.project_key IS NULL AND previous.project_key IS NOT NULL) AS no_longer_present_count,
        COUNT_IF(
            current.project_key IS NOT NULL
            AND previous.project_key IS NOT NULL
            AND NOT (current.psgc_region_code <=> previous.psgc_region_code)
        ) AS region_changed_count
    FROM current_projects AS current
    FULL OUTER JOIN previous_projects AS previous
        ON current.project_key = previous.project_key
    GROUP BY COALESCE(current.psgc_region_code, previous.psgc_region_code)
)

SELECT
    region.region_key,
    dim_region.psgc_region_code,
    region.region_name,
    region.region_short_name,
    region.source_coverage_status,
    CASE
        WHEN context.previous_source_snapshot_id IS NULL THEN 'NO_PRIOR_SNAPSHOT'
        ELSE 'COMPARABLE'
    END AS comparison_status,
    context.current_source_snapshot_id,
    context.current_snapshot_date_key,
    context.previous_source_snapshot_id,
    context.previous_snapshot_date_key,
    COALESCE(current.project_count, 0) AS current_project_count,
    COALESCE(current.finished_count, 0) AS current_finished_count,
    COALESCE(current.active_count, 0) AS current_active_count,
    COALESCE(current.inactive_count, 0) AS current_inactive_count,
    COALESCE(current.unmapped_status_count, 0) AS current_unmapped_status_count,
    ROUND(100.0 * COALESCE(current.finished_count, 0) / NULLIF(COALESCE(current.project_count, 0), 0), 2)
        AS current_finished_pct,
    ROUND(100.0 * COALESCE(current.active_count, 0) / NULLIF(COALESCE(current.project_count, 0), 0), 2)
        AS current_active_pct,
    ROUND(100.0 * COALESCE(current.inactive_count, 0) / NULLIF(COALESCE(current.project_count, 0), 0), 2)
        AS current_inactive_pct,
    ROUND(100.0 * COALESCE(current.unmapped_status_count, 0) / NULLIF(COALESCE(current.project_count, 0), 0), 2)
        AS current_unmapped_status_pct,
    current.total_reported_budget_pesos AS current_total_reported_budget_pesos,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(previous.project_count, 0) END
        AS previous_project_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(previous.finished_count, 0) END
        AS previous_finished_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(previous.active_count, 0) END
        AS previous_active_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(previous.inactive_count, 0) END
        AS previous_inactive_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(previous.unmapped_status_count, 0) END
        AS previous_unmapped_status_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN previous.total_reported_budget_pesos END
        AS previous_total_reported_budget_pesos,
    CASE
        WHEN context.previous_source_snapshot_id IS NOT NULL
            THEN COALESCE(current.project_count, 0) - COALESCE(previous.project_count, 0)
    END AS project_count_change,
    CASE
        WHEN context.previous_source_snapshot_id IS NOT NULL
            AND current.total_reported_budget_pesos IS NOT NULL
            AND previous.total_reported_budget_pesos IS NOT NULL
            THEN current.total_reported_budget_pesos - previous.total_reported_budget_pesos
    END AS reported_budget_change_pesos,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.status_changed_count, 0) END
        AS status_changed_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.moved_to_finished_count, 0) END
        AS moved_to_finished_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.progress_increased_count, 0) END
        AS progress_increased_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.progress_decreased_count, 0) END
        AS progress_decreased_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.new_project_count, 0) END
        AS new_project_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.no_longer_present_count, 0) END
        AS no_longer_present_count,
    CASE WHEN context.previous_source_snapshot_id IS NOT NULL THEN COALESCE(changes.region_changed_count, 0) END
        AS region_changed_count
FROM `05-analytics`.analytics_regional_investment AS region
INNER JOIN `03-gold`.dim_region AS dim_region
    ON region.region_key = dim_region.region_key
CROSS JOIN snapshot_context AS context
LEFT JOIN current_summary AS current
    ON dim_region.psgc_region_code = current.psgc_region_code
LEFT JOIN previous_summary AS previous
    ON dim_region.psgc_region_code = previous.psgc_region_code
LEFT JOIN project_changes AS changes
    ON dim_region.psgc_region_code = changes.reporting_region_code
ORDER BY region.region_name;

## Summary

**What:** Preview the regional rows and confirm whether historical comparison is available.

**Why:** `NO_PRIOR_SNAPSHOT` is expected in a fresh workspace with only one processed DPWH snapshot. It is not a zero-change result.

In [ ]:
%sql
SELECT *
FROM `05-analytics`.analytics_region_project_progress
ORDER BY region_name;